In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.preprocessing import StandardScaler
from umap import UMAP
import hdbscan
from tqdm import tqdm
import gc
import time
import warnings
import kDBCV
from itertools import product
import scipy

warnings.filterwarnings("ignore")

C:\Users\filipe.figueira\OneDrive - SEF-MG\Área de Trabalho\MBA\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Carrega o banco de dados

df = pd.read_csv('dados_pre_processados_8708.csv')

In [3]:
# Funções auxiliares para as modelagens:

def carrega_embeddings(caminho_embeddings: str, indice: set, variavel: str) -> np.ndarray:
    """
    Carregas os embeddings salvos em formato .parquet, em que há, neste banco de dados, uma variável de índice.

    Args:
        caminho_embeddings (str): A pasta de origem em que o embedding está salvo;
        indice (str): O nome da variável de índice;
        variavel (str): O nome da variável que foi transformada em embedding.

    Returns:
        np.ndarray: Uma matriz de tamanho 1 x dimensão do embedding, contendo os embeddings carregados
    """
    
    nome_variavel_embedding = f'embedding_{variavel}'
    tbl = pq.read_table(caminho_embeddings, columns = ['idx', nome_variavel_embedding]).to_pandas()
    tbl = tbl[tbl['idx'].isin(indice)].sort_values('idx')
    return np.vstack(tbl[nome_variavel_embedding].apply(np.array)).astype(np.float32)

def gerar_amostra(dados: np.ndarray, rotulos: np.ndarray, tamanho_amostra: int, semente_aleatoria: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Seleciona uma amostra estratificada e reprodutível de um conjunto de dados e seus rótulos.
    A estratificação mantém a proporção de outliers (rótulo -1) e não-outliers igual à do dataset.

    Args:
        dados (np.ndarray): O array de dados (vetores de características, ex: X_umap);
        rotulos (np.ndarray): O array com os rótulos de cluster correspondentes;
        tamanho_amostra (int): O número de pontos desejado na amostra;
        semente_aleatoria (int): A semente para o gerador de números aleatórios.

    Returns:
        tuple[np.ndarray, np.ndarray]: (dados_da_amostra, rotulos_da_amostra).
    """
    n_total_pontos = len(rotulos)

    # Caso o dataset seja menor ou igual ao tamanho da amostra, retorna o dataset completo
    if n_total_pontos <= tamanho_amostra:
        return dados, rotulos

    rng = np.random.default_rng(semente_aleatoria)

    # Máscaras para outliers e não-outliers
    mask_outliers = (rotulos == -1)
    mask_inliers = ~mask_outliers

    n_outliers = mask_outliers.sum()
    n_inliers = mask_inliers.sum()

    # Proporção de outliers no dataset
    prop_outliers = n_outliers / n_total_pontos

    # Número de outliers e inliers a sortear na amostra
    n_outliers_amostra = int(round(prop_outliers * tamanho_amostra))
    n_inliers_amostra = tamanho_amostra - n_outliers_amostra

    # Sorteio estratificado sem reposição
    idx_outliers = rng.choice(np.where(mask_outliers)[0], size = min(n_outliers_amostra, n_outliers), replace = False)
    idx_inliers = rng.choice(np.where(mask_inliers)[0], size = min(n_inliers_amostra, n_inliers), replace = False)

    # Junta e embaralha os índices
    idx_final = np.concatenate([idx_outliers, idx_inliers])
    rng.shuffle(idx_final)

    return dados[idx_final], rotulos[idx_final]

# Modelagem, em que são testados os 30 modelos com UMAP + HDBSCAN:

In [4]:
modelos_de_linguagem = {'SBERT':'embeddings_parquet', 'SBERT multilíngue':'embeddings_parquet_multilingual'}
variaveis_textuais = ['xprod', 'T1', 'T2', 'T3', 'T4']
variaveis_numericas = ['vuncom', 'log_vuncom']

lista_experimentos = []

# Loop 1: Itera sobre os modelos de linguagem (SBERT, SBERT multilíngue):
for nome_linguagem, diretorio in modelos_de_linguagem.items():
    
    # Loop 2: Itera sobre as variáveis de texto (xprod, T1, T2, ...):
    for variavel in variaveis_textuais:
        
        # Combinação 1: Apenas o embedding de texto:
        lista_experimentos.append({
            'modelo_linguagem': nome_linguagem,
            'diretorio_embedding': diretorio,
            'variavel_texto': variavel,
            'variavel_numerica': None, # Usamos None para indicar ausência
            'nome_descritivo': f'{variavel} (somente embedding)'
        })
        
        # Combinação 2 e 3: Embedding + cada uma das variáveis numéricas:
        for var_num in variaveis_numericas:
            nome_descritivo_num = f"{variavel} + {var_num.replace('_', '')}"
            lista_experimentos.append({
                'modelo_linguagem': nome_linguagem,
                'diretorio_embedding': diretorio,
                'variavel_texto': variavel,
                'variavel_numerica': var_num,
                'nome_descritivo': nome_descritivo_num
            })

# --- Criação do DataFrame final ---
df_experimentos = pd.DataFrame(lista_experimentos)

# Cria a lista de resultados finais:
lista_resultados_finais = []

progress_bar = tqdm(df_experimentos.iterrows(), total = df_experimentos.shape[0], desc = "Executando Experimentos")

for index, row in progress_bar:

    

    # Seleciona as variáveis para o vetor de características:
    coluna_texto_atual = row['variavel_texto']
    var_num = row['variavel_numerica']
    variavel_numerica_atual = None if var_num == None else var_num
    
    if variavel_numerica_atual != None:
        combinacao_variaveis = [coluna_texto_atual, variavel_numerica_atual]
    else:
        combinacao_variaveis = [coluna_texto_atual]
    
    # Define o subgrupo:
    df_subgrupo = df.drop_duplicates(subset = combinacao_variaveis).sort_values('idx').reset_index(drop = True)
    indices = set(df_subgrupo['idx'].values)
    
    # Define os embeddings:
    caminho_arquivo_embedding = f'{row['diretorio_embedding']}/embedding_{coluna_texto_atual}_8708.parquet'
    
    # Carrega os embeddings:
    embedding_texto = carrega_embeddings(caminho_arquivo_embedding, indices, coluna_texto_atual)
    
    if variavel_numerica_atual != None:
        # Se houver variável numérica, forma o vetor de características:
        vuncom_numerico = df_subgrupo[variavel_numerica_atual].fillna(0).values.reshape(-1, 1)
        vuncom_normalizado = StandardScaler().fit_transform(vuncom_numerico)
        X = np.hstack([embedding_texto, vuncom_normalizado]).astype(np.float32)
    else:
        X = embedding_texto
    
    inicio_tempo = time.perf_counter()
    
    # UMAP:
    umap_model = UMAP(n_neighbors = 15, min_dist = 0.1, n_components = 30, metric = 'cosine', low_memory = True, random_state = 1234)
    X_umap = umap_model.fit_transform(X)
    
    # HDBSCAN:
    hdb_model = hdbscan.HDBSCAN(min_cluster_size = 100, min_samples = 20, metric = 'euclidean', core_dist_n_jobs = 1)
    rotulos = hdb_model.fit_predict(X_umap)

    tempo_execucao = round(time.perf_counter() - inicio_tempo, 2)

    # Amostragem:    
    X_amostra, rotulos_amostra = gerar_amostra(dados = X_umap, rotulos = rotulos, tamanho_amostra = 5000, semente_aleatoria = 1234)

    # Calcula o DBCV:
    score_dbcv = None
    qtd_clusters_amostra = len(set(rotulos_amostra)) - (1 if -1 in rotulos_amostra else 0)

    # O DBCV precisa de no mínimo 2 clusters para ter um resultado, por isso inicia em None e o teste é feito.
    if qtd_clusters_amostra >= 2:
        try:
            score_dbcv = kDBCV.DBCV_score(X_amostra, rotulos_amostra)
            if score_dbcv:
                dbcv = score_dbcv[0]
        except:
            dbcv = -99 # Valor que acusa erro no cálculo do dbcv
    else:
        dbcv = None

    # Cálculo das outras métricas no dataset completo:
    qtd_clusters = len(set(rotulos)) - (1 if -1 in rotulos else 0)
    qtd_outliers = int((rotulos == -1).sum())
    qtd_linhas_total = len(df_subgrupo)
    percentual_outliers = (qtd_outliers / qtd_linhas_total) * 100 if qtd_linhas_total > 0 else 0

    lista_resultados_finais.append({
        'Modelo de linguagem': row['modelo_linguagem'], 
        'modelo': row['nome_descritivo'], 
        'dbcv': dbcv,
        'qtd_clusters': qtd_clusters, 
        'qtd_outliers': qtd_outliers,
        'percentual_outliers': percentual_outliers,
        'qtd_linhas': qtd_linhas_total, 
        'tempo_exec_seg': tempo_execucao
    })
    
    # Limpeza de memória para a próxima iteração:
    del X, X_umap, hdb_model, rotulos, df_subgrupo, X_amostra, rotulos_amostra
    if 'embedding_texto' in locals():
        del embedding_texto
    gc.collect()

# Transforma a lista de resultados em um DataFrame do Pandas:
df_resultados_finais = pd.DataFrame(lista_resultados_finais)

# Salva o resultado em CSV
df_resultados_finais.to_csv('resultados_30_modelos_8708.csv')

Executando Experimentos: 100%|█████████████████████████████████████████████████████████| 30/30 [46:35<00:00, 93.18s/it]


In [7]:
# Todos os 8 primeiros modelos possuem dbcv maior que 0,39. O primeiro modelo "SBERT - T4 + log_vuncom" possui DBCV
# igual a 0,399498. Por outro lado, o modelo "SBERT multilíngue - T4" possui DBCV DE 0,390586, que é apenas 2,2308%
# menor, mas seu tempo de execução em segundos também é 40,7218% menor, fazendo diferença em bases de dados escaláveis.

df_resultados_finais.sort_values(by = 'dbcv', ascending = False).head(10)

,Modelo de linguagem,modelo,dbcv,qtd_clusters,qtd_outliers,percentual_outliers,qtd_linhas,tempo_exec_seg
14,SBERT,T4 + logvuncom,0.399498,96,9059,19.180606,47230,98.08
11,SBERT,T3 + logvuncom,0.395604,97,9522,20.015555,47573,110.99
24,SBERT multilíngue,T3 (somente embedding),0.394778,77,11086,27.424302,40424,68.25
25,SBERT multilíngue,T3 + vuncom,0.394771,89,13143,27.627015,47573,90.08
7,SBERT,T2 + vuncom,0.392895,91,9666,20.317394,47575,117.78
12,SBERT,T4 (somente embedding),0.392803,74,6491,18.147506,35768,75.73
9,SBERT,T3 (somente embedding),0.392293,86,7818,19.339996,40424,84.63
27,SBERT multilíngue,T4 (somente embedding),0.390586,64,7140,19.961977,35768,58.14
10,SBERT,T3 + vuncom,0.384058,87,9666,20.318248,47573,115.89
8,SBERT,T2 + logvuncom,0.377179,90,9707,20.403573,47575,109.55


# Escolha dos hiperparâmetros do UMAP e do HDBSCAN:

In [9]:
 # Seleciona as variáveis para o vetor de características:
coluna_texto_atual = 'T4'
pasta_embeddings = 'embeddings_parquet_multilingual'

# Define o subgrupo:
df_subgrupo = df.drop_duplicates(subset = coluna_texto_atual).sort_values('idx').reset_index(drop = True)
indices = set(df_subgrupo['idx'].values)
n = df_subgrupo.shape[0]
    
# Define os embeddings:
caminho_arquivo_embedding = f'{pasta_embeddings}/embedding_{coluna_texto_atual}_8708.parquet'
    
# Carrega os embeddings:
embedding_texto = carrega_embeddings(caminho_arquivo_embedding, indices, coluna_texto_atual)
    
# Montando o vetor de características:
X = embedding_texto.astype(np.float32)

# Gridsearch:

# Hiperparâmetros
UMAP_GRID = {'n_neighbors': [15, 30, 50, 80], 'min_dist': [0.0, 0.1, 0.25, 0.5], 'n_components': [5, 15, 30, 50, 80]}
HDBSCAN_GRID = {'min_cluster_size': [50, 100, 200, 300], 'min_samples': [10, 25, 35, 50, 80]}
umap_grid = list(product(UMAP_GRID['n_neighbors'], UMAP_GRID['min_dist'], UMAP_GRID['n_components']))
hdbscan_grid = list(product(HDBSCAN_GRID['min_cluster_size'], HDBSCAN_GRID['min_samples']))
full_grid = list(product(umap_grid, hdbscan_grid))

resultados = []
progress_bar_umap = tqdm(umap_grid, desc = 'GridSearch UMAP', ncols = 100)
t_inicial = time.perf_counter()

for umap_params in progress_bar_umap:
    
    n_neighbors, min_dist, n_components = umap_params

    # Rodar cada UMAP uma vez por conjunto de parâmetros:
    umap_model = UMAP(n_neighbors = n_neighbors, min_dist = min_dist, n_components = n_components,
                      metric = 'cosine', low_memory = True, random_state = 1234)
    X_umap = umap_model.fit_transform(X)

    progress_bar_hdbscan = tqdm(hdbscan_grid, desc = 'GridSearch HDBSCAN', ncols = 80, leave = False)

    for hdbscan_params in progress_bar_hdbscan:
        
        min_cluster_size, min_samples = hdbscan_params

        inicio_tempo = time.perf_counter()

        # Rodar cada HDBSCAN para cada UMAP:
        hdb = hdbscan.HDBSCAN(min_cluster_size = min_cluster_size, min_samples = min_samples,
                          metric='euclidean', cluster_selection_method = 'eom').fit(X_umap)
        rotulos = hdb.labels_

        # Contabiliza o tempo para rodar o HDBCSAN:
        tempo = round(time.perf_counter() - inicio_tempo, 2)

        # Gera a amostra estratificada:
        X_amostra, rotulos_amostra = gerar_amostra(dados = X_umap, rotulos = rotulos, tamanho_amostra = 5000, semente_aleatoria = 1234)

        # Calcula o DBCV:
        score_dbcv = None
        qtd_clusters_amostra = len(set(rotulos_amostra)) - (1 if -1 in rotulos_amostra else 0)

        # O DBCV precisa de no mínimo 2 clusters para ter um resultado, por isso inicia em None e o teste é feito.
        if qtd_clusters_amostra >= 2:
            try:
                score_dbcv = kDBCV.DBCV_score(X_amostra, rotulos_amostra)
                if score_dbcv:
                    dbcv = score_dbcv[0]
            except:
                dbcv = -99 # Valor que acusa erro no cálculo do dbcv
        else:
            dbcv = None
                
        # Demais métricas:    
        qtd_clusters = len(set(rotulos)) - (1 if -1 in rotulos else 0)
        qtd_outliers = int((rotulos == -1).sum())
    
        resultados.append({
            'n_neighbors': n_neighbors, 'min_dist': min_dist, 'n_components': n_components,
            'min_cluster_size': min_cluster_size, 'min_samples': min_samples,
            'dbcv': dbcv, 'qtd_clusters': qtd_clusters, 'qtd_outliers': qtd_outliers,
            'tempo_exec': tempo, 'perc_outlier': qtd_outliers / n})
    
        progress_bar_hdbscan.set_postfix(dbcv = f'{dbcv or 0:.4f}', clusters = qtd_clusters)

        del hdb, rotulos, X_amostra, rotulos_amostra
        gc.collect()

    del X_umap
    gc.collect()

t_final = time.perf_counter() - t_inicial

# Transforma a lista de resultados em um DataFrame do Pandas:
resultados = pd.DataFrame(resultados)

# Salva o resultado em CSV
resultados.to_csv('resultado_grid_search.csv')
print(f'Professo finalizado.')
print(f'Tempo: {t_final}')

GridSearch HDBSCAN:   0%|     | 0/20 [00:12<?, ?it/s, clusters=144, dbcv=0.3967]
GridSearch HDBSCAN:   5%| | 1/20 [00:12<03:59, 12.61s/it, clusters=144, dbcv=0.3
GridSearch HDBSCAN:   5%| | 1/20 [00:13<03:59, 12.61s/it, clusters=135, dbcv=0.4
GridSearch HDBSCAN:  10%| | 2/20 [00:13<01:46,  5.94s/it, clusters=135, dbcv=0.4
GridSearch HDBSCAN:  10%| | 2/20 [00:15<01:46,  5.94s/it, clusters=116, dbcv=0.4
GridSearch HDBSCAN:  15%|▏| 3/20 [00:15<01:05,  3.87s/it, clusters=116, dbcv=0.4
GridSearch HDBSCAN:  15%|▏| 3/20 [00:16<01:05,  3.87s/it, clusters=118, dbcv=0.4
GridSearch HDBSCAN:  20%|▏| 4/20 [00:16<00:45,  2.83s/it, clusters=118, dbcv=0.4
GridSearch HDBSCAN:  20%|▏| 4/20 [00:17<00:45,  2.83s/it, clusters=80, dbcv=0.45
GridSearch HDBSCAN:  25%|▎| 5/20 [00:17<00:34,  2.33s/it, clusters=80, dbcv=0.45
GridSearch HDBSCAN:  25%|▎| 5/20 [00:18<00:34,  2.33s/it, clusters=75, dbcv=0.32
GridSearch HDBSCAN:  30%|▎| 6/20 [00:19<00:26,  1.92s/it, clusters=75, dbcv=0.32
GridSearch HDBSCAN:  30%|▎| 

Professo finalizado.
Tempo: 40885.20205260004


In [35]:
# O melhor modelo foi: SBERT multilíngue - T4 com os hiperparâmetros iguais a:
# n_neighbors=30, min_dist=0, n_components=50, min_cluster_size=50, min_samples=80

resultados[resultados.qtd_clusters > 10].sort_values(by = 'dbcv', ascending = False).head(10)

# Modelagem com o melhor modelo encontrado:

In [7]:
# Seleciona as variáveis para o vetor de características:
coluna_texto_atual = 'T4'
pasta_embeddings = 'embeddings_parquet_multilingual'
n_neighbors = 30
min_dist = 0.0
n_components = 50
min_cluster_size = 50
min_samples = 80

resultado_modelo_final = []

# Define o subgrupo:
df_subgrupo = df.drop_duplicates(subset = coluna_texto_atual).sort_values('idx').reset_index(drop = True)
indices = set(df_subgrupo['idx'].values)
    
# Define os embeddings:
caminho_arquivo_embedding = f'{pasta_embeddings}/embedding_{coluna_texto_atual}_8708.parquet'
    
# Carrega os embeddings:
embedding_texto = carrega_embeddings(caminho_arquivo_embedding, indices, coluna_texto_atual)

X = embedding_texto.astype(np.float32)
    
inicio_tempo = time.perf_counter()
    
# UMAP:
umap_model = UMAP(n_neighbors = n_neighbors, min_dist = min_dist, n_components = n_components, metric = 'cosine', low_memory = True, random_state = 1234)
X_umap = umap_model.fit_transform(X)
    
# HDBSCAN:
hdb_model = hdbscan.HDBSCAN(min_cluster_size = min_cluster_size, min_samples = min_samples, metric = 'euclidean', core_dist_n_jobs = 1)
rotulos = hdb_model.fit_predict(X_umap)

# Atribui os rótulos ao df_subgrupo, que é aquele com itens únicos:
df_subgrupo['clusters'] = rotulos

tempo_execucao = round(time.perf_counter() - inicio_tempo, 2)

# Calcula o DBCV:
score_dbcv = None
try:
    score_dbcv = kDBCV.DBCV_score(X_umap, rotulos)
    if score_dbcv:
        dbcv = score_dbcv[0]
except:
    dbcv = -99 # Valor que acusa erro no cálculo do dbcv

# Cálculo das outras métricas no dataset completo:
qtd_clusters = len(set(rotulos)) - (1 if -1 in rotulos else 0)
qtd_outliers = int((rotulos == -1).sum())
qtd_linhas_total = len(df_subgrupo)
percentual_outliers = (qtd_outliers / qtd_linhas_total) * 100 if qtd_linhas_total > 0 else 0

resultado_modelo_final.append({
    'dbcv': dbcv,
    'qtd_clusters': qtd_clusters,
    'qtd_outliers': qtd_outliers,
    'percentual_outliers': percentual_outliers,
    'qtd_linhas': qtd_linhas_total,
    'tempo_exec_seg': tempo_execucao
})

# Salva o resultado em CSV
df_subgrupo.to_csv('dados_pre_processados_8708_com_clusters.csv')
print(resultado_modelo_final)

[{'dbcv': 0.5466890625114472, 'qtd_clusters': 74, 'qtd_outliers': 6762, 'percentual_outliers': 18.9051666293894, 'qtd_linhas': 35768, 'tempo_exec_seg': 132.35}]
